In [1]:
!pip install requests

In [8]:
import requests

# -------------------------------
# MITRE ATT&CK Mapping Dictionary
# -------------------------------
mitre_mapping = {
    "remote code execution": {
        "technique": "T1190 - Exploit Public-Facing Application",
        "tactic": "Initial Access"
    },
    "privilege escalation": {
        "technique": "T1068 - Exploitation for Privilege Escalation",
        "tactic": "Privilege Escalation"
    },
    "authentication bypass": {
        "technique": "T1190 - Exploit Public-Facing Application",
        "tactic": "Initial Access"
    },
    "command execution": {
        "technique": "T1059 - Command and Scripting Interpreter",
        "tactic": "Execution"
    }
}

# -------------------------------
# Function to Fetch CVE Data
# -------------------------------
def fetch_cve_data(cve_id):
    url = f"https://services.nvd.nist.gov/rest/json/cves/2.0?cveId={cve_id}"
    response = requests.get(url)

    if response.status_code != 200:
        return None

    data = response.json()

    try:
        cve_data = data["vulnerabilities"][0]["cve"]
        description = cve_data["descriptions"][0]["value"]

        metrics = cve_data.get("metrics", {})
        if "cvssMetricV31" in metrics:
            cvss_score = metrics["cvssMetricV31"][0]["cvssData"]["baseScore"]
        elif "cvssMetricV30" in metrics:
            cvss_score = metrics["cvssMetricV30"][0]["cvssData"]["baseScore"]
        else:
            cvss_score = "Not Available"

        return description, cvss_score

    except:
        return None

# -------------------------------
# Function to Map to MITRE
# -------------------------------
def map_to_mitre(description):
    description_lower = description.lower()

    for keyword in mitre_mapping:
        if keyword in description_lower:
            return mitre_mapping[keyword]

    return {
        "technique": "Not Mapped",
        "tactic": "Not Mapped"
    }

# -------------------------------
# Main Program
# -------------------------------
cve_id = input("Enter CVE ID (example: CVE-2021-3156): ")

result = fetch_cve_data(cve_id)

if result:
    description, cvss_score = result
    mitre_result = map_to_mitre(description)

    print("\n===== CVE DETAILS =====")
    print("Description:", description)
    print("CVSS Score:", cvss_score)

    print("\n===== MITRE ATT&CK Mapping =====")
    print("Technique:", mitre_result["technique"])
    print("Tactic:", mitre_result["tactic"])
else:
    print("CVE not found or error retrieving data.")

Enter CVE ID (example: CVE-2021-3156): CVE-2025-12345

===== CVE DETAILS =====
Description: A security vulnerability has been detected in LLM-Claw 0.1.0/0.1.1/0.1.1a/0.1.1a-p1. The affected element is the function agent_deploy_init of the file /agents/deploy/initiate.c of the component Agent Deployment. Such manipulation leads to buffer overflow. It is possible to launch the attack remotely. A patch should be applied to remediate this issue.
CVSS Score: 8.8

===== MITRE ATT&CK Mapping =====
Technique: Not Mapped
Tactic: Not Mapped
